# Consolidated Homing/Escape Detection Pipeline

Clean, streamlined workflow for Phase 1-3 homing detection analysis.  
Pipeline for learning features of manually labelled homings and applying them to a new homing detection logic

In [11]:
%reload_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path

from behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated import HomingAnalyzer
from settings.settings_analyze_behave import Settings

%matplotlib inline

In [13]:
# Import experiments
from behave_analysis.database.Experiments.JAL004_ex import (
    JAL4_flip4_3Sept, JAL4_flip6_19Sept, JAL4_flip3_28aug, JAL4_flip5_11Sept
)
from behave_analysis.database.Experiments.JAL005_ex import JAL5_flip1_8Sept, JAL5_flip3_21Sept
from behave_analysis.database.Experiments.JAL006_ex import (
    JAL6_flip6_28mar, JAL6_flip4_21mar, JAL6_flip3_18mar, JAL6_flip5_25mar
)
from behave_analysis.database.Experiments.JAL007_ex import (
    JAL7_flip8_9apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_flip9_16apr, JAL7_flip10_23apr
)
from behave_analysis.database.Experiments.JAL008_ex import (
    JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip3_7may, JAL8_flip5_14may, JAL8_flip4_10may
)

experiments_objects = [JAL4_flip4_3Sept,
    JAL4_flip6_19Sept,
    JAL4_flip3_28aug,
    JAL4_flip5_11Sept,
    JAL5_flip1_8Sept,
    JAL5_flip3_21Sept,
    JAL6_flip6_28mar,
    JAL6_flip4_21mar,
    JAL6_flip3_18mar,
    JAL6_flip5_25mar,
    JAL7_flip8_9apr,
    JAL7_flip5_22mar,
    JAL7_flip2_12mar,
    JAL7_flip9_16apr,
    JAL7_flip10_23apr,
    JAL8_flip1_25apr,
    JAL8_flip2_29apr,
    JAL8_flip3_7may,
    JAL8_flip5_14may,
    JAL8_flip4_10may,
]

In [14]:
# Initialize and load data

analyzer = HomingAnalyzer(experiments_objects, Settings)
analyzer.load_all_sessions()
print(f"✓ Loaded {len(analyzer.session_data)} sessions")

2026-06-18 13:37:59.417 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated:load_all_sessions:103 - Loading 20 sessions...
2026-06-18 13:37:59.422 | INFO     | behave_analysis.database.computer_ID:get_computer_specific_paths:47 - All data has been moved from winstor to ceph so we should always load ceph data now
2026-06-18 13:38:30.268 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated:load_all_sessions:133 -   ✓ JAL004_flip_2023_09_03
2026-06-18 13:38:30.273 | INFO     | behave_analysis.database.computer_ID:get_computer_specific_paths:47 - All data has been moved from winstor to ceph so we should always load ceph data now
2026-06-18 13:39:04.491 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated:load_all_sessions:133 -   ✓ JAL004_flip_2023_09_19
2026-06-18 13:39:04.497 | INFO     | behave_analysis.database.computer_ID:get_computer_specific_paths:47 - All data has been

✓ Loaded 20 sessions


In [15]:
# Phase 1: Extract runs and manual events
speed_threshold = Settings.homings_speed_threshold
speed_gap_tolerance = Settings.homings_gap_tolerance
analyzer.extract_runs(speed_threshold=speed_threshold, gap_tolerance_frames=speed_gap_tolerance)
analyzer.extract_manual_runs()
analyzer.label_extracted_runs(overlap_threshold=0.5)

print(f"✓ Extracted {len(analyzer.extracted_runs)} runs")
print(f"✓ Extracted {len(analyzer.manual_runs)} manually-labelled events")

n_exploration = sum(1 for r in analyzer.extracted_runs if r.is_exploration)
print(f"  - {n_exploration} exploration runs")
print(f"  - {len(analyzer.extracted_runs) - n_exploration} target-overlapping runs")

2026-06-18 13:51:25.147 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated:extract_runs:164 - Extracting runs (threshold=4.0 cm/s, gap_tolerance=1 frames)...
2026-06-18 13:51:26.206 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated:extract_runs:203 -   ✓ Extracted 92503 runs across 20 sessions
2026-06-18 13:51:26.206 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated:extract_manual_runs:207 - Extracting manually-labelled homing/escape runs...
2026-06-18 13:51:26.530 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated:extract_manual_runs:238 -   ✓ Extracted 1908 manually-labelled runs
2026-06-18 13:51:26.530 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated:label_extracted_runs:266 - Labeling extracted runs (overlap_threshold=0.5)...
2026-06-18 13:51:57.129 | INFO     | behave_analysis.ana

✓ Extracted 92503 runs
✓ Extracted 1908 manually-labelled events
  - 89902 exploration runs
  - 2601 target-overlapping runs


In [16]:
# Extract features for all runs
analyzer.extract_features()
print(f"✓ Extracted features for {len(analyzer.all_runs)} runs")

2026-06-18 13:52:04.570 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated:extract_features:290 - Extracting features for all runs...
2026-06-18 13:52:16.708 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated:extract_features:304 -   ✓ Extracted features for 94411 runs


✓ Extracted features for 94411 runs


In [18]:
# Compute and plot feature distributions (exploration vs manual target)
discrimination_results = analyzer.compute_feature_distributions()
analyzer.plot_feature_distributions(only_top_n=len(analyzer.feature_ranking))
plt.show()

print("\n✓ Top discriminative features:")
for idx, (fname, stats_dict) in enumerate(analyzer.feature_ranking):
    print(
        f"  {idx+1}. {fname}: d={stats_dict['cohens_d']:.3f}, "
        f"AUC_pooled={stats_dict['auc']:.3f}, "
        f"AUC_session_mean={stats_dict['auc_session_mean']:.3f} "
        f"(n_sessions={stats_dict['auc_session_n']})"
    )

2026-06-18 13:56:53.885 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated:compute_feature_distributions:381 - Computing feature distributions (exploration vs manually-labelled)...
2026-06-18 13:56:56.990 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated:compute_feature_distributions:458 -   Feature ranking (by |Cohen's d|):
2026-06-18 13:56:56.990 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated:compute_feature_distributions:460 -     1. net_distance: d=3.146, AUC=0.970, p=0.000e+00
2026-06-18 13:56:56.990 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated:compute_feature_distributions:460 -     2. net_dy: d=3.118, AUC=0.991, p=0.000e+00
2026-06-18 13:56:57.003 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated:compute_feature_distributions:460 -     3. speed_peak: d=2.278, AUC=0.975, 


✓ Top discriminative features:
  1. net_distance: d=3.146, AUC_pooled=0.970, AUC_session_mean=0.969 (n_sessions=20)
  2. net_dy: d=3.118, AUC_pooled=0.991, AUC_session_mean=0.990 (n_sessions=20)
  3. speed_peak: d=2.278, AUC_pooled=0.975, AUC_session_mean=0.980 (n_sessions=20)
  4. speed_mean: d=1.991, AUC_pooled=0.932, AUC_session_mean=0.929 (n_sessions=20)
  5. displacement_vertical_ratio: d=1.868, AUC_pooled=0.876, AUC_session_mean=0.876 (n_sessions=20)
  6. initial_hdir_change_abs: d=1.220, AUC_pooled=0.810, AUC_session_mean=0.813 (n_sessions=20)
  7. mean_initial_acceleration: d=0.915, AUC_pooled=0.785, AUC_session_mean=0.777 (n_sessions=20)
  8. net_dx: d=-0.248, AUC_pooled=0.424, AUC_session_mean=0.438 (n_sessions=20)
  9. mean_initial_angular_head_velocity: d=0.210, AUC_pooled=0.664, AUC_session_mean=0.676 (n_sessions=20)
  10. head_turn_angle_initial: d=0.192, AUC_pooled=0.555, AUC_session_mean=0.564 (n_sessions=20)


In [22]:
# Phase 2: Fit gates from manual distributions OR use manual thresholds
# Option A: Learn gates from manually-labelled data (default)
gates = analyzer.fit_classification_gates()

print(f"\n✓ Fitted {len(gates)} feature gates")
for fname, gate in gates.items():
    print(f"  {fname}: {gate['dir']} {gate['threshold']:.4f} (recall={gate['recall']:.3f}, precision={gate['precision']:.3f})")

2026-06-18 15:47:41.984 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated:fit_classification_gates:573 - Fitting gates (target_recall=0.9)...
2026-06-18 15:47:42.080 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated:fit_classification_gates:641 -   net_distance: keep if value >= 36.4551 (recall=0.910, precision=0.203)
2026-06-18 15:47:42.158 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated:fit_classification_gates:641 -   net_dy: keep if value >= 23.7879 (recall=0.926, precision=0.435)
2026-06-18 15:47:42.227 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated:fit_classification_gates:641 -   speed_peak: keep if value >= 30.5942 (recall=0.911, precision=0.225)
2026-06-18 15:47:42.297 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated:fit_classification_gates:641 -   speed_mean: keep if 


✓ Fitted 10 feature gates
  net_distance: >= 36.4551 (recall=0.910, precision=0.203)
  net_dy: >= 23.7879 (recall=0.926, precision=0.435)
  speed_peak: >= 30.5942 (recall=0.911, precision=0.225)
  speed_mean: >= 12.1877 (recall=0.905, precision=0.089)
  displacement_vertical_ratio: >= 0.5706 (recall=0.905, precision=0.077)
  initial_hdir_change_abs: >= 0.7365 (recall=0.900, precision=0.159)
  mean_initial_acceleration: >= 5.3728 (recall=0.901, precision=0.080)
  net_dx: <= 53.8980 (recall=0.995, precision=0.021)
  mean_initial_angular_head_velocity: >= 0.0139 (recall=0.901, precision=0.027)
  head_turn_angle_initial: >= -2.1063 (recall=0.901, precision=0.022)


In [26]:
# Run Phase 2 classification
candidates = analyzer.run_classification(use_learned_gates=True)

print(f"\n✓ Phase 2 classification complete")
print(f"Total candidates: {len(candidates)}")

# Count per session
for session_name, cands in analyzer.candidates_by_session.items():
    print(f"  {session_name}: {len(cands)} candidates")

2026-06-18 15:50:06.079 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated:run_classification:685 -   Loaded 10 gates; 3 pass filters (recall>=0.9, precision>=0.1, auc>=0.9, |d|>=1)
2026-06-18 15:50:06.088 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated:run_classification:692 - Running Phase 2 classification with 3 gates...
2026-06-18 15:50:06.265 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated:run_classification:728 -   ✓ 2468 runs passed Phase 2 gates



✓ Phase 2 classification complete
Total candidates: 2468
  JAL004_flip_2023_09_03: 76 candidates
  JAL004_flip_2023_09_19: 93 candidates
  JAL004_flip_2023_08_28: 47 candidates
  JAL004_flip_2023_09_11: 31 candidates
  JAL005_flip_puff1_8th_sept_jal005_2023_09_08: 70 candidates
  JAL005_21stSept_barrierflip_2023_09_21: 36 candidates
  JAL006_flip_2024_03_28: 211 candidates
  JAL006_flip_2024_03_21: 177 candidates
  JAL006_flip_2024_03_18: 195 candidates
  JAL006_flip_2024_03_25: 143 candidates
  JAL007_sesh8_2024_04_09: 146 candidates
  JAL007_sesh8_2024_03_22: 110 candidates
  JAL007_sesh8_2024_03_12: 108 candidates
  JAL007_sesh8_2024_04_16: 105 candidates
  JAL007_sesh8_2024_04_23: 132 candidates
  JAL008_flip_2024_04_25: 120 candidates
  JAL008_flip_2024_04_29: 71 candidates
  JAL008_flip_2024_05_7: 194 candidates
  JAL008_flip_2024_05_14: 204 candidates
  JAL008_flip_2024_05_10: 199 candidates


In [ ]:
# Phase 3: Overlap metrics
results = analyzer.compute_manualvsauto_overlap(
    candidates_by_session=analyzer.candidates_by_session
 )

print(f"\n✓ Phase 3 overlap analysis:")
for session_name, metrics in results['by_session'].items():
    print(f"\n  {session_name}:")
    print(f"    Extracted runs: {metrics['n_extracted']}")
    print(f"    Manual events: {metrics['n_manual']}")
    print(f"    Mean frac labelled: {metrics['mean_fraction_extracted_that_are_manually_labelled']:.3f}")
    print(f"    Mean frac extracted: {metrics['mean_fraction_manual_that_are_extracted']:.3f}")
    print(f"    Manual with >1 extracted: {metrics['fraction_manual_with_multiple_extracted']:.3f}")

2026-06-18 15:50:47.680 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated:compute_manualvsauto_overlap:764 - Computing Phase 3 overlap metrics...
2026-06-18 15:50:48.393 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.analyze_homing_consolidated:compute_manualvsauto_overlap:838 -   ✓ Phase 3 overlap computed



✓ Phase 3 overlap analysis:

  JAL004_flip_2023_09_03:
    Extracted runs: 76
    Manual events: 75
    Mean frac labelled: 0.750
    Mean frac extracted: 0.612
    Manual with >1 extracted: 0.053

  JAL004_flip_2023_09_19:
    Extracted runs: 93
    Manual events: 94
    Mean frac labelled: 0.712
    Mean frac extracted: 0.489
    Manual with >1 extracted: 0.011

  JAL004_flip_2023_08_28:
    Extracted runs: 47
    Manual events: 34
    Mean frac labelled: 0.398
    Mean frac extracted: 0.444
    Manual with >1 extracted: 0.029

  JAL004_flip_2023_09_11:
    Extracted runs: 31
    Manual events: 50
    Mean frac labelled: 0.854
    Mean frac extracted: 0.393
    Manual with >1 extracted: 0.000

  JAL005_flip_puff1_8th_sept_jal005_2023_09_08:
    Extracted runs: 70
    Manual events: 79
    Mean frac labelled: 0.641
    Mean frac extracted: 0.444
    Manual with >1 extracted: 0.025

  JAL005_21stSept_barrierflip_2023_09_21:
    Extracted runs: 36
    Manual events: 58
    Mean frac la

In [ ]:
# 3,11,14,15,19
# Launch interactive Syd viewer with curation
# NOTE: You can inspect candidate and manual masks in all modes
# include_manual_events_in_iteration=True  -> iterate over candidates + manual
# include_manual_events_in_iteration=False -> iterate candidates only (still shows both masks)

viewer, removed_runs = analyzer.create_syd_viewer(
    candidates=analyzer.candidates_by_session,
    session_name=None,  # None = all sessions; set to specific name to filter
    include_manual_events_in_iteration=False,
    manual_curation = False, # this can only be done when you load the homings dict!
 )
viewer.show()

c:\Users\Jasmine\miniconda3\envs\JAL2pipeline\lib\site-packages\syd\notebook_deployment\deployer.py:55: UserWarning: The current backend (other) is not supported. Please use %matplotlib widget or %matplotlib inline.
The behavior of the viewer will almost definitely not work as expected!
  warnings.warn(
